In [5]:
import pandas as pd
import ast

df = pd.read_csv('../data/raw/track_data_final.csv')
df['release_year'] = pd.to_datetime(df['album_release_date'], errors='coerce').dt.year
df_last_10_years = df[df['release_year'] >= 2016].copy()

# 2. 
def parse_genres(genre_val):
    if pd.isna(genre_val) or genre_val == 'N/A' or genre_val == '[]':
        return ['unknown']
    if isinstance(genre_val, list):
        return genre_val if genre_val else ['unknown']
    try:
        parsed = ast.literal_eval(genre_val)
        if isinstance(parsed, list) and len(parsed) > 0:
            return parsed
    except:
        pass
    cleaned = [g.strip(" '\"[]") for g in str(genre_val).split(',') if g.strip()]
    return cleaned if cleaned else ['unknown']

# 3. exploded dataframe
df_temp = df_last_10_years.copy()
df_temp['parsed_genres'] = df_temp['artist_genres'].apply(parse_genres)
df_exploded = df_temp.explode('parsed_genres')



# 1. Define the high level categories:
def map_to_high_level_genre(genre_str):
    genre_lower = str(genre_str).lower()
    
    # roles for keywords
    if any(k in genre_lower for k in ['pop', 'dance pop', 'boy band']):
        return 'Pop'
    elif any(k in genre_lower for k in ['rap', 'hip hop', 'trap', 'drill', 'melodic rap']):
        return 'Hip-Hop / Rap'
    elif any(k in genre_lower for k in ['rock', 'metal', 'grunge', 'punk', 'alternative']):
        return 'Rock / Metal'
    elif any(k in genre_lower for k in ['latin', 'reggaeton', 'urbano latino', 'trap latino', 'sierraño', 'corridos']):
        return 'Latin'
    elif any(k in genre_lower for k in ['edm', 'house', 'techno', 'electro', 'trance', 'dance']):
        return 'Electronic / EDM'
    elif any(k in genre_lower for k in ['r&b', 'soul', 'urban contemporary', 'quiet storm']):
        return 'R&B / Soul'
    elif any(k in genre_lower for k in ['indie', 'indie pop', 'bedroom pop', 'folk']):
        return 'Indie / Alternative'
    elif any(k in genre_lower for k in ['classical', 'ambient', 'instrumental', 'soundtrack']):
        return 'Classical / Instrumental'
    else:
        return 'Other / Niche'

# 2. Apply the function to the parsed/exploded genres

df_exploded['high_level_genre'] = df_exploded['parsed_genres'].apply(map_to_high_level_genre)

# 3. Let’s check the result: how many songs fall into each of the main categories?
print(df_exploded['high_level_genre'].value_counts())


# Filter out the rows that fall into the 'Other / Niche' category.
other_genres = df_exploded[df_exploded['high_level_genre'] == 'Other / Niche']

# Check the most common ones or list the unique original genres from them
print("List of uncategorized (Other / Niche) original genres:")
print(other_genres['parsed_genres'].value_counts().head(30))

high_level_genre
Other / Niche               3818
Pop                          866
Hip-Hop / Rap                736
Electronic / EDM             307
Latin                        277
Classical / Instrumental     180
Rock / Metal                 179
R&B / Soul                   172
Indie / Alternative           44
Name: count, dtype: int64
A kategorizálatlan (Other / Niche) eredeti műfajok listája:
parsed_genres
Unknown             3026
country              112
mahraganat            54
anime                 52
medieval              36
acoustic country      32
nightcore             25
celtic                18
southern gothic       18
darkwave              13
big room              13
uk grime              12
neo-psychedelic       12
afrobeats             12
emo                   12
grime                 11
khaleeji              11
egyptian shaabi        9
future bass            9
melodic bass           9
corrido                8
classic country        8
brazilian bass         7
phonk      